In [6]:
import xarray as xr
import numpy as np
import pandas as pd
from libpysal.weights import lat2W
from esda import Moran
np.random.seed(12345)

In [7]:
def extract_year_slice(ds, varname, year):
    da = ds[varname].sel(time=f"{year}-01-01")
    return da.values

In [8]:
def compute_moran(arr2d):
    H, W = arr2d.shape
    
    # NaN 填零（libpysal 需要）
    flat = np.nan_to_num(arr2d.flatten(), nan=0.0)
    
    w = lat2W(H, W)
    w.transform = "r"
    
    mi = Moran(flat, w, permutations=0)
    return mi.I


In [9]:
def compute_all_variables(ds, years, output_csv):
    names = ["agri","grassland","forest"] 
    modes = ["basin","region"]

    variables = [f"{m}_{n}" for n in names for m in modes]

    long_results = []

    for varname in variables:
        print(f"\n=== Processing variable: {varname} ===")

        for year in years:
            print(f"Starting {year}")
            arr2d = extract_year_slice(ds, varname, year)
            mi = compute_moran(arr2d)

            long_results.append({
                "year": year,
                "variable": varname,
                "moran": mi
            })

    df_long = pd.DataFrame(long_results)

    df_wide = df_long.pivot(index="year", columns="variable", values="moran")

    df_wide.to_csv(output_csv)
    print(f"Saved CSV to {output_csv}")

    return df_wide


In [10]:
years = list(range(2010, 2101, 10))
years.insert(0, 2005)

ds = xr.open_dataset(f"../../NC/compare.nc")

df = compute_all_variables(ds, years, output_csv=f"../../CSV/moran/world_moran.csv")
df



=== Processing variable: basin_agri ===
Starting 2005
Starting 2010
Starting 2020
Starting 2030
Starting 2040
Starting 2050
Starting 2060
Starting 2070
Starting 2080
Starting 2090
Starting 2100

=== Processing variable: region_agri ===
Starting 2005
Starting 2010
Starting 2020
Starting 2030
Starting 2040
Starting 2050
Starting 2060
Starting 2070
Starting 2080
Starting 2090
Starting 2100

=== Processing variable: basin_grassland ===
Starting 2005
Starting 2010
Starting 2020
Starting 2030
Starting 2040
Starting 2050
Starting 2060
Starting 2070
Starting 2080
Starting 2090
Starting 2100

=== Processing variable: region_grassland ===
Starting 2005
Starting 2010
Starting 2020
Starting 2030
Starting 2040
Starting 2050
Starting 2060
Starting 2070
Starting 2080
Starting 2090
Starting 2100

=== Processing variable: basin_forest ===
Starting 2005
Starting 2010
Starting 2020
Starting 2030
Starting 2040
Starting 2050
Starting 2060
Starting 2070
Starting 2080
Starting 2090
Starting 2100

=== Proces

variable,basin_agri,basin_forest,basin_grassland,region_agri,region_forest,region_grassland
year,,,,,,
2005,0.888334,0.856536,0.929444,0.899068,0.831726,0.927462
2010,0.882408,0.856270,0.929430,0.894182,0.830160,0.927467
2020,0.870641,0.855626,0.926769,0.889058,0.830527,0.925978
2030,0.865688,0.855467,0.921471,0.885511,0.831253,0.923156
2040,0.860471,0.854349,0.917934,0.884100,0.831964,0.921353
2050,0.854260,0.853311,0.916666,0.879901,0.831812,0.920300
2060,0.848483,0.852484,0.915333,0.875205,0.831295,0.919394
2070,0.845386,0.851915,0.914302,0.869504,0.830420,0.919220
2080,0.842094,0.851198,0.913020,0.861240,0.829552,0.919565
